# Tradepad
Option valuation on one underlying, OVME-style: define the deal, pick a model (Black-Scholes closed forms or local
vol), see the market inputs and the valuation (same functions as the web app, `tqstudio/views/tradepad.py`).

## Deal and model

In [ ]:
from datetime import date

TICKER = "TICKER1"
PRODUCT = "Vanilla"            # Vanilla | Digital | Barrier
OPTION_TYPE = "Put"            # Call | Put
STRIKE_PCT = 80.0              # strike / spot, %
EXPIRY = date(2027, 11, 5)
QUANTITY = 1.0
# Digital
PAYOFF, CASH_AMOUNT = "Cash", 100.0          # Cash | Asset
# Barrier (continuous monitoring)
BARRIER_TYPE, BARRIER_PCT = "DownIn", 70.0   # DownIn | DownOut | UpIn | UpOut

MODEL = "Local vol"            # Black-Scholes | Local vol
N_PATHS = 50_000               # local vol Monte Carlo

In [ ]:
import os
import sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "marketdata").is_dir())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorquant as tq
from tqstudio.views.tradepad import ModelCache, price_trade

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
cache = ModelCache()                                               # calibrated local vols, reused across pricings
trade = dict(ticker=TICKER, product=PRODUCT, option_type=OPTION_TYPE, strike_pct=STRIKE_PCT,
             expiry=EXPIRY.isoformat(), quantity=QUANTITY, payoff=PAYOFF, cash_amount=CASH_AMOUNT,
             barrier_type=BARRIER_TYPE, barrier_pct=BARRIER_PCT)

## Valuation

In [ ]:
out = price_trade(mkt, cache, trade, MODEL, n_paths=N_PATHS)
t, r = out["trade"], out["result"]
eq = mkt.get_equity(TICKER)

underlying = {"ticker": TICKER, "currency": eq.currency.name, "snapshot": mkt.evaluation_date,
              "spot": round(t["spot"], 2)}
deal = {"product": PRODUCT, "type": OPTION_TYPE, "strike": f"{t['strike']:.2f} ({STRIKE_PCT:g}%)",
        "expiry": f"{t['expiry']} ({t['T']:.2f}y)", "quantity": QUANTITY}
if PRODUCT == "Digital":
    deal["payoff"] = PAYOFF + (f" {CASH_AMOUNT:g}" if PAYOFF == "Cash" else "")
if PRODUCT == "Barrier":
    deal["barrier"] = f"{BARRIER_TYPE} {t['barrier']:.2f} ({BARRIER_PCT:g}%), continuous"
market = {"forward": round(t["forward"], 2), "rate (zero, %)": round(100 * t["rate"], 4),
          "discount factor": round(t["discount_factor"], 5), "repo (%)": round(100 * t["repo"], 4),
          "PV dividends": round(t["pv_dividends"], 4), "implied div yield (%)": round(100 * t["implied_div_yield"], 4),
          "vol at strike (%)": round(100 * t["vol_at_strike"], 2), "vol ATM forward (%)": round(100 * t["vol_atm_forward"], 2)}
model = {"model": MODEL}
if t["n_paths"]:
    model["Monte Carlo"] = f"{t['n_paths']:,} paths, {t['n_steps']} steps"
valuation = {"price": round(r["price"], 4), "premium (% spot)": round(100 * r["unit_price"] / t["spot"], 4),
             "s.e.": None if r["standard_error"] is None else round(r["standard_error"], 4),
             "delta": round(r["delta"], 4), "delta (%)": round(100 * r["delta"] / QUANTITY, 2),
             "vega (1 vol pt)": round(r["vega"], 4), "rho (1 bp)": round(r["rho"], 4),
             "calibration [s]": round(r["calibration_s"], 2), "pricing [s]": round(r["pricing_s"], 2)}
panels = {"Underlying": underlying, "Deal": deal, "Market": market, "Model": model, "Valuation": valuation}
pd.concat({k: pd.Series(v, dtype=object) for k, v in panels.items()}).to_frame("value")

- **price**: per unit times the quantity; **premium**: unit price as % of spot; **s.e.**: Monte Carlo standard error.
- **delta**: dV/dS (shares); **vega**: sum of the vega per surface node, per vol point; **rho**: sum over the
  discount-curve pillars, per bp.
- Black-Scholes: closed forms at the vol of the strike (bilinear on the surface); digitals as a call / put
  spread on the smile (strikes K +- 0.5%), the desk standard (N(d2) at the vol of the strike misses the skew). Local vol: Monte Carlo on the local
  vol calibrated on the first use per underlying (then cached), greeks by AAD to the surface nodes.

## Vega per surface node

In [ ]:
grid = out["vega_grid"]
v = np.asarray(r["vega_nodes"])
scale = np.abs(v).max()
fig, ax = plt.subplots(figsize=(7, 3.8))
im = ax.imshow(v, cmap="RdBu_r", vmin=-scale, vmax=scale, aspect="auto")
for (i, j), x in np.ndenumerate(v):
    if abs(x) >= 0.01 * scale:
        ax.text(j, i, f"{x:.3f}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(len(grid["moneyness"])), [f"{100 * m:.0f}" for m in grid["moneyness"]], fontsize=8)
ax.set_yticks(range(len(grid["expiries"])), [f"{x:.2f}" for x in grid["expiries"]], fontsize=8)
ax.set(xlabel="K / S0 %", ylabel="T (years)", title=f"{MODEL}: vega per surface node (1 vol point)")
fig.colorbar(im, ax=ax, label="vega");

## Same deal, the other model

In [ ]:
other = "Black-Scholes" if MODEL == "Local vol" else "Local vol"
r2 = price_trade(mkt, cache, trade, other, n_paths=N_PATHS)["result"]
pd.DataFrame([{k: x[k] for k in ("model", "price", "standard_error", "delta", "vega", "rho", "calibration_s",
                                 "pricing_s")} for x in (r, r2)]).set_index("model").round(4)